In [1]:
pip install xgboost joblib pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import joblib
import tarfile
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
import numpy as np
from scipy.stats import entropy
from sklearn.preprocessing import MinMaxScaler

In [5]:
df = pd.read_csv('tng_mock_transactions (1).csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2363 entries, 0 to 2362
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   idx                 2363 non-null   int64  
 1   transaction_id      2363 non-null   object 
 2   user_id             2363 non-null   object 
 3   transaction_date    2363 non-null   object 
 4   product_category    2363 non-null   object 
 5   merchant_name       2363 non-null   object 
 6   product_amount      2363 non-null   float64
 7   transaction_status  2363 non-null   object 
 8   merchant_id         2363 non-null   object 
 9   transaction_type    2363 non-null   object 
dtypes: float64(1), int64(1), object(8)
memory usage: 184.7+ KB


In [7]:
#feature engineering
# Load data
df['transaction_date'] = pd.to_datetime(df['transaction_date'], dayfirst=True)
df['year_month'] = df['transaction_date'].dt.to_period('M')

# Reference date for recency (day after last txn)
ref_date = df['transaction_date'].max() + pd.Timedelta(days=1)

def engineer_pillars(group):
    # Setup
    payments = group[group['transaction_type'] == 'payment']
    topups = group[group['transaction_type'] == 'topup']
    months_active = group['year_month'].nunique()
    date_range = (group['transaction_date'].max().to_period('M') - group['transaction_date'].min().to_period('M')).n + 1

    # 🔵 Pillar 1: Top-up Consistency (20%)
    f_topup_freq = len(topups) / months_active if months_active > 0 else 0 #1: topup frequency = total topups / active months
    f_topup_consistency = topups['year_month'].nunique() / date_range #2: topup consistency = topup active months / total account months
    f_avg_topup_amt = topups['product_amount'].mean() if not topups.empty else 0 #3: avg topup amount = total topup amount / number of topups
    if not topups.empty and len(topups.groupby('year_month')) > 1:
        topup_monthly = topups.groupby('year_month')['product_amount'].sum()
        cv = topup_monthly.std() / (topup_monthly.mean() + 1e-9)
        f_topup_stability = 1 / (1 + cv) #4: topup stability = 1 / (1 + CV of monthly topup amounts)
    else:
        f_topup_stability = 0
    f_topup_to_spend = topups['product_amount'].sum() / (payments['product_amount'].sum() + 1e-9) #5: topup to spend ratio = total topup amount / total payment amount

    # 🟠 Pillar 2: Spending Behaviour (25%)
    f_avg_monthly_spend = payments['product_amount'].sum() / months_active if months_active > 0 else 0
    if not payments.empty and len(payments.groupby('year_month')) > 1:
        spend_monthly = payments.groupby('year_month')['product_amount'].sum()
        cv_s = spend_monthly.std() / (spend_monthly.mean() + 1e-9)
        f_spend_stability = 1 / (1 + cv_s)
    else:
        f_spend_stability = 0
    f_avg_payment_amt = payments['product_amount'].mean() if not payments.empty else 0

    # 🟡 Pillar 3: Transaction Frequency (20%)
    f_avg_monthly_txn = len(payments) / months_active if months_active > 0 else 0
    f_active_ratio = months_active / date_range
    f_recency = (ref_date - group['transaction_date'].max()).days
    f_txn_regularity = payments.groupby('year_month').size().std() if len(payments.groupby('year_month')) > 1 else 0

    # 🟢 Pillar 4: Merchant Diversity (15%)
    f_unique_cats = group['product_category'].nunique()
    cat_dist = group['product_category'].value_counts(normalize=True)
    f_cat_entropy = entropy(cat_dist)

    # 🔴 Pillar 5: Reliability & Risk (15%)
    f_failed_rate = (group['transaction_status'] == 'Failed').mean()
    # f_pending_rate = (group['transaction_status'] == 'Pending').mean()
    f_success_rate = (group['transaction_status'] == 'Successful').mean()

    # ⚪ Pillar 6: History Maturity (5%)
    f_account_age = date_range

    # Metadata
    is_spend_only = 1 if len(topups) == 0 else 0
    raw_txn_count = len(group)

    return pd.Series([
        f_topup_freq,f_topup_consistency, f_avg_topup_amt, f_topup_stability, f_topup_to_spend,
        f_avg_monthly_spend, f_spend_stability, f_avg_payment_amt,
        f_avg_monthly_txn, f_active_ratio, f_recency, f_txn_regularity,
        f_unique_cats, f_cat_entropy,
        f_failed_rate, f_success_rate,
        f_account_age, is_spend_only, raw_txn_count
    ])

In [8]:

columns = [
    'topup_frequency', 'topup_consistency', 'avg_topup_amount', 'topup_stability', 'topup_to_spend_ratio',
    'avg_monthly_spend', 'spend_stability', 'avg_payment_amount',
    'avg_monthly_txn_count', 'active_months_ratio', 'recency_days', 'txn_regularity',
    'unique_categories', 'category_entropy',
    'failed_rate', 'success_rate',
    'account_age', 'is_spend_only', 'raw_txn_count'
]

In [9]:
#run transformation
user_features = df.groupby('user_id').apply(engineer_pillars).reset_index()
user_features.columns = ['user_id'] + columns

C:\Users\adham\AppData\Local\Temp\ipykernel_7456\203783808.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_features = df.groupby('user_id').apply(engineer_pillars).reset_index()


In [10]:
user_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                6 non-null      object 
 1   topup_frequency        6 non-null      float64
 2   topup_consistency      6 non-null      float64
 3   avg_topup_amount       6 non-null      float64
 4   topup_stability        6 non-null      float64
 5   topup_to_spend_ratio   6 non-null      float64
 6   avg_monthly_spend      6 non-null      float64
 7   spend_stability        6 non-null      float64
 8   avg_payment_amount     6 non-null      float64
 9   avg_monthly_txn_count  6 non-null      float64
 10  active_months_ratio    6 non-null      float64
 11  recency_days           6 non-null      float64
 12  txn_regularity         6 non-null      float64
 13  unique_categories      6 non-null      float64
 14  category_entropy       6 non-null      float64
 15  failed_rat

In [11]:
# 1. Normalize
scaler = MinMaxScaler()
feat_to_scale = [c for c in columns if c not in ['is_spend_only', 'raw_txn_count']]
user_features_norm = user_features.copy()
user_features_norm[feat_to_scale] = scaler.fit_transform(user_features[feat_to_scale])

# 2. Invert Negative Signals
inverse_cols = ['recency_days', 'txn_regularity', 'failed_rate']
for col in inverse_cols:
    user_features_norm[col] = 1 - user_features_norm[col]

# 3. Calculate Pillar Sub-scores
user_features['score_topup'] = user_features_norm[['topup_frequency','topup_consistency', 'avg_topup_amount', 'topup_stability', 'topup_to_spend_ratio']].mean(axis=1)
user_features['score_spending'] = user_features_norm[['avg_monthly_spend', 'spend_stability', 'avg_payment_amount']].mean(axis=1)
user_features['score_frequency'] = user_features_norm[['avg_monthly_txn_count', 'active_months_ratio', 'recency_days', 'txn_regularity']].mean(axis=1)
user_features['score_diversity'] = user_features_norm[['unique_categories', 'category_entropy']].mean(axis=1)
user_features['score_reliability'] = user_features_norm[['failed_rate', 'success_rate']].mean(axis=1)
user_features['score_maturity'] = user_features_norm['account_age']

# 4. Final Weighted Score with Spend-Only Adjustment
def calculate_final_score(row):
    if row['is_spend_only'] == 1:
        # Redistribute Topup(20%) to Spending(+10%) and Reliability(+10%)
        weights = {'spending': 0.35, 'frequency': 0.20, 'diversity': 0.15, 'reliability': 0.25, 'maturity': 0.05}
        raw_score = (row['score_spending'] * weights['spending'] +
                     row['score_frequency'] * weights['frequency'] +
                     row['score_diversity'] * weights['diversity'] +
                     row['score_reliability'] * weights['reliability'] +
                     row['score_maturity'] * weights['maturity'])
    else:
        weights = {'topup': 0.20, 'spending': 0.25, 'frequency': 0.20, 'diversity': 0.15, 'reliability': 0.15, 'maturity': 0.05}
        raw_score = (row['score_topup'] * weights['topup'] +
                     row['score_spending'] * weights['spending'] +
                     row['score_frequency'] * weights['frequency'] +
                     row['score_diversity'] * weights['diversity'] +
                     row['score_reliability'] * weights['reliability'] +
                     row['score_maturity'] * weights['maturity'])


    return 300 + (raw_score * 550)

user_features['credit_score'] = user_features.apply(calculate_final_score, axis=1).round().astype(int)

# Categorize
def get_tier(score):
    if score >= 750: return 'Excellent'
    if score >= 650: return 'Very Good'
    if score >= 550: return 'Good'
    if score >= 450: return 'Fair'
    return 'Poor'

user_features['risk_tier'] = user_features['credit_score'].apply(get_tier)

In [12]:
user_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 28 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                6 non-null      object 
 1   topup_frequency        6 non-null      float64
 2   topup_consistency      6 non-null      float64
 3   avg_topup_amount       6 non-null      float64
 4   topup_stability        6 non-null      float64
 5   topup_to_spend_ratio   6 non-null      float64
 6   avg_monthly_spend      6 non-null      float64
 7   spend_stability        6 non-null      float64
 8   avg_payment_amount     6 non-null      float64
 9   avg_monthly_txn_count  6 non-null      float64
 10  active_months_ratio    6 non-null      float64
 11  recency_days           6 non-null      float64
 12  txn_regularity         6 non-null      float64
 13  unique_categories      6 non-null      float64
 14  category_entropy       6 non-null      float64
 15  failed_rat

In [13]:
df = user_features.copy()

In [15]:
#model

import joblib
import pandas as pd
import tarfile
import xgboost as xgb

# 1. Extract the model file from the archive
with tarfile.open('xgboost_model.tar.gz', 'r:gz') as tar:
    tar.extractall()
    # Assuming the extracted file is 'xgboost_model.joblib'

# 2. Load the model
model = joblib.load('xgboost_model.joblib')



# 4. Prepare the Features
# Identify the features used during training.
# Based on your CSV, we exclude 'user_id' (identifier) and existing labels ('risk_tier', 'is_risk').
feature_cols = [
    'topup_frequency', 'topup_consistency', 'avg_topup_amount', 'topup_stability',
    'topup_to_spend_ratio', 'avg_monthly_spend', 'spend_stability', 'avg_payment_amount',
    'avg_monthly_txn_count', 'active_months_ratio', 'recency_days', 'txn_regularity',
     'category_entropy', 'failed_rate', 'success_rate',
    'account_age', 'is_spend_only', 'raw_txn_count'
]

X = df[feature_cols]

# 5. Generate Predictions
# This gets the binary risk prediction (0 or 1)
df['predicted_risk'] = model.predict(X)

# 6. Map to Risk Labels (Optional)
# If your model predicts 'is_risk', you can map it back to labels:
df['risk_label'] = df['predicted_risk'].map({0: 'Low Risk', 1: 'High Risk'})

# Save the results
df[['user_id', 'predicted_risk', 'risk_label','credit_score','risk_tier','score_spending','score_frequency','score_diversity','score_reliability','score_maturity']].to_csv('predictions_output.csv', index=False)
print("Predictions saved to predictions_output.csv")

[23:19:41] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, please export the model by calling `Booster.save_model` from that version
  first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/latest/tutorials/saving_model.html

  for more details about differences between saving model and serializing.



C:\Users\adham\AppData\Local\Temp\ipykernel_7456\2705727241.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


AttributeError: 'XGBClassifier' object has no attribute 'use_label_encoder'

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3927 entries, 0 to 3926
Data columns (total 30 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                3927 non-null   object 
 1   topup_frequency        3927 non-null   float64
 2   topup_consistency      3927 non-null   float64
 3   avg_topup_amount       3927 non-null   float64
 4   topup_stability        3927 non-null   float64
 5   topup_to_spend_ratio   3927 non-null   float64
 6   avg_monthly_spend      3927 non-null   float64
 7   spend_stability        3927 non-null   float64
 8   avg_payment_amount     3927 non-null   float64
 9   avg_monthly_txn_count  3927 non-null   float64
 10  active_months_ratio    3927 non-null   float64
 11  recency_days           3927 non-null   float64
 12  txn_regularity         3927 non-null   float64
 13  unique_categories      3927 non-null   float64
 14  category_entropy       3927 non-null   float64
 15  fail